# HYRON RVC Cloud Trainer v0.5.12

Este notebook verifica o ambiente Kaggle, confirma disponibilidade de GPU e exige confirmacao explicita antes de qualquer treinamento.

Sem dataset autorizado e sem GPU, o fluxo para antes de qualquer etapa de treino.

In [ ]:
from __future__ import annotations

import os
import platform
import shutil
import sys
from pathlib import Path

if "torch" in sys.modules:
    raise RuntimeError("Restart the kernel before running diagnostics so GPU 0 is selected before PyTorch initializes.")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TORCH_FORCE_WEIGHTS_ONLY_LOAD"] = "1"
print("CUDA_VISIBLE_DEVICES set to physical GPU 0 before importing torch")
print("Restricted weights-only checkpoint loading enforced for this process")

try:
    import psutil
except Exception:  # pragma: no cover
    psutil = None

try:
    import torch
except Exception:  # pragma: no cover
    torch = None

print("Kaggle kernel type:", os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "not-set"))
print("Python:", platform.python_version())
print("Platform:", platform.platform())
print("Executable:", sys.executable)


In [ ]:
def memory_info_mb():
    if psutil is None:
        return {"total_ram_mb": None, "available_ram_mb": None}
    mem = psutil.virtual_memory()
    return {"total_ram_mb": mem.total / (1024 * 1024), "available_ram_mb": mem.available / (1024 * 1024)}

mem = memory_info_mb()
print("RAM total:", mem["total_ram_mb"])
print("RAM available:", mem["available_ram_mb"])

disk = shutil.disk_usage("/")
print("Disk total GB:", disk.total / (1024 ** 3))
print("Disk free GB:", disk.free / (1024 ** 3))

if torch is not None:
    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("Visible GPU count:", torch.cuda.device_count())
        print("Selected visible device:", 0)
        print("GPU name:", torch.cuda.get_device_name(0))
        print("GPU total memory MB:", torch.cuda.get_device_properties(0).total_memory / (1024 ** 2))
        free_vram, total_vram = torch.cuda.mem_get_info(0)
        print("GPU free memory MB:", free_vram / (1024 ** 2))
        print("GPU total memory from CUDA MB:", total_vram / (1024 ** 2))
        print("CUDA version:", torch.version.cuda)
    else:
        print("GPU is unavailable in this session.")
else:
    print("torch is not installed; GPU detection is unavailable.")


## Veredito do ambiente

Se faltarem os requisitos abaixo, o notebook deve informar claramente o bloqueio e interromper antes do treinamento:

In [ ]:
warnings = []
if torch is None:
    warnings.append("PyTorch not installed")
if torch is not None and not torch.cuda.is_available():
    warnings.append("GPU CUDA unavailable")
if psutil is not None:
    available_ram = psutil.virtual_memory().available / (1024 ** 2)
    if available_ram < 3500:
        warnings.append(f"RAM available is too low for a conservative RVC run: {available_ram:.0f} MB")
disk = shutil.disk_usage("/")
if disk.free < 10 * (1024 ** 3):
    warnings.append("Less than 10 GB of free disk space; training may fail")

if warnings:
    print("BLOCKED:")
    for item in warnings:
        print("-", item)
else:
    print("Resource checks passed; RVC runtime compatibility is NOT verified.")
    print("Do not train until compatibility, dataset authorization, and explicit operator confirmation are established.")


In [ ]:
dataset_candidates = [
    Path("/kaggle/input"),
    Path("/kaggle"),
    Path.cwd(),
]
valid = []
for candidate in dataset_candidates:
    if candidate.exists():
        valid.extend(sorted(([p for p in candidate.iterdir() if p.is_dir()]))[:20])
print("Candidate dataset directories:")
for item in valid:
    print("-", item)

print("\nRecommended paths:")
print("- Prepared dataset:", "/kaggle/working/rvc-prepared")
print("- Export package:", "/kaggle/working/rvc-export")
print("- Config:", "/kaggle/working/tools/rvc-cloud-trainer/configs/training.json")
print("\nIMPORTANT: the notebook must not start training automatically. Use a confirmed dataset and explicit authorization first.")


## Compatibilidade do runtime fixado

O commit fixado inclui um perfil de dependencias `requirments_cu128_py312.txt`, rotulado para Python 3.12, que especifica PyTorch/Torchaudio 2.7.1+cu128 e exige Torch abaixo de 2.8; tambem fixa `numpy<2`. Esse conjunto nao corresponde ao ambiente Kaggle Python 3.13 / PyTorch 2.11 e nao deve ser instalado. A verificacao abaixo e apenas diagnostica: nao instala, atualiza ou rebaixa pacotes.

In [ ]:
import importlib.metadata
import inspect

print("Runtime compatibility status: NOT VERIFIED for training")
print("Python:", platform.python_version())
print("PyTorch:", getattr(torch, "__version__", "not installed"))
if torch is not None:
    print("torch.load supports weights_only:", "weights_only" in inspect.signature(torch.load).parameters)

for package in ("numpy", "librosa", "faiss-cpu", "av", "praat-parselmouth", "scipy", "soundfile", "transformers", "tensorboard", "matplotlib", "ffmpeg-python", "torchaudio"):
    try:
        print(f"{package}=={importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package}: MISSING")

print("Do not run pip install -r requirments_cu128_py312.txt in this Kaggle session.")
print("Do not install or downgrade torch/torchaudio automatically.")

## Dependencias do caminho RVC (sem instalar)

**Obrigatorios no fluxo de treino/preprocessamento:** PyTorch, NumPy, SciPy, librosa, soundfile, TensorBoard, matplotlib, Transformers (para HuBERT e seus assets), PyAV (`av`) e ffmpeg-python. `infer.audio` importa PyAV no topo, entao a ausencia de `av` bloqueia esse caminho de audio. `praat-parselmouth` e importado no topo do extrator F0, mesmo quando a configuracao escolhe RMVPE; portanto, este checkout bloqueia o import/uso do extrator sem ele. `faiss-cpu` e opcional se nao gerar/usar indice. `torchaudio` e opcional para o backend CUDA de resample: `infer.audio` tem fallback FFmpeg. Dependencias da UI/ONNX nao sao necessarias ao caminho minimo de treino.

Estas classificacoes descrevem imports/caminhos encontrados no commit selecionado; os testes abaixo determinarao o que realmente importa nesta sessao. Nenhuma dependencia sera instalada.

In [ ]:
import subprocess

print("Read-only dependency consistency check: python -m pip check")
pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True,
    check=False,
)
print(pip_check.stdout or "(no stdout)")
if pip_check.stderr:
    print("stderr:", pip_check.stderr)
print("pip check exit code:", pip_check.returncode)
print("This command checks installed metadata only; it does not install or change packages.")

declared_bounds = {
    "python": "upstream profile explicitly targets Python 3.12",
    "torch": "upstream profile requires torch 2.7.1+cu128 and <2.8",
    "numpy": ">=1.26.4,<2",
    "librosa": ">=0.10.2,<0.11",
    "transformers": ">=4.49.0,<4.50",
}
observed_versions = {
    "python": platform.python_version(),
    "torch": str(getattr(torch, "__version__", "missing")).split("+", 1)[0],
}
for package in ("numpy", "librosa", "transformers"):
    try:
        observed_versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        observed_versions[package] = "missing"
print("Observed versions:", observed_versions)
print("Upstream declared bounds:", declared_bounds)
print("Python/Torch/NumPy/librosa/Transformers outside the declared profile are not evidence of runtime failure; their RVC compatibility remains unverified until smoke tests and pipeline validation pass.")

## Checkout, sintaxe e patches (somente leitura)

Defina `RVC_ROOT` para o checkout Kaggle do runtime. A celula nao clona, nao altera arquivos e nao aplica patches. Para testar patches, disponibilize o script revisado `harden_runtime.py` do HYRON num Kaggle Dataset privado e informe o caminho abaixo. Sao chamadas de verificacao (`check_commit`, `verify_expected_worktree` e auditoria AST), nunca `harden()`.

In [ ]:
import importlib.util
EXPECTED_RVC_COMMIT = "81eed5e8f68b6bed1789f682fe78cdd324495afc"
RVC_ROOT = Path(os.environ.get("HYRON_RVC_ROOT", "/kaggle/working/Retrieval-based-Voice-Conversion-WebUI"))
HYRON_HARDENER = Path(os.environ.get("HYRON_HARDENER_PATH", "/kaggle/input/hyron-rvc-tools/harden_runtime.py"))

if not RVC_ROOT.is_dir():
    print(f"CHECKOUT: BLOCKED; runtime directory not found: {RVC_ROOT}")
else:
    revision = subprocess.run(["git", "-C", str(RVC_ROOT), "rev-parse", "HEAD"], capture_output=True, text=True, check=False)
    actual_commit = revision.stdout.strip()
    print("Expected commit:", EXPECTED_RVC_COMMIT)
    print("Actual commit:", actual_commit or revision.stderr.strip())
    print("CHECKOUT:", "PASS" if revision.returncode == 0 and actual_commit == EXPECTED_RVC_COMMIT else "FAIL")

    if not HYRON_HARDENER.is_file():
        print(f"PATCH AUDIT: NOT RUN; provide reviewed helper at {HYRON_HARDENER}")
    elif revision.returncode == 0 and actual_commit == EXPECTED_RVC_COMMIT:
        spec = importlib.util.spec_from_file_location("hyron_harden_runtime_readonly", HYRON_HARDENER)
        if spec is None or spec.loader is None:
            print("PATCH AUDIT: FAIL; could not load hardening helper")
        else:
            hardener = importlib.util.module_from_spec(spec)
            sys.dont_write_bytecode = True
            spec.loader.exec_module(hardener)
            try:
                hardener.check_commit(RVC_ROOT)
                hardener.verify_expected_worktree(RVC_ROOT)
                unsafe_loaders, faiss_loaders = hardener.inspect_other_loaders(RVC_ROOT)
                if unsafe_loaders:
                    raise RuntimeError("additional deserializers require review: " + ", ".join(unsafe_loaders))
                print("PATCH AUDIT: PASS; exact expected worktree patches and weights_only=True loaders verified")
                print("FAISS loader files (not imported/used in this check):", faiss_loaders)
            except Exception as error:
                print(f"PATCH AUDIT: FAIL; {type(error).__name__}: {error}")

    syntax_roots = [RVC_ROOT / "train", RVC_ROOT / "infer", RVC_ROOT / "configs"]
    syntax_files = [path for root in syntax_roots if root.is_dir() for path in root.rglob("*.py")]
    syntax_errors = []
    for path in syntax_files:
        try:
            compile(path.read_text(encoding="utf-8"), str(path), "exec")
        except (SyntaxError, UnicodeError, OSError) as error:
            syntax_errors.append((str(path), f"{type(error).__name__}: {error}"))
    print(f"SYNTAX: {len(syntax_files) - len(syntax_errors)}/{len(syntax_files)} Python files compile without executing or writing bytecode")
    for path, error in syntax_errors:
        print("SYNTAX FAIL:", path, error)

print("No checkpoint was opened; no training/preprocessing entry point was invoked.")

## Smoke tests de importacao (isolados, sem treino)

Cada modulo e importado num processo Python separado com cwd/PYTHONPATH apontando para o checkout. `PYTHONDONTWRITEBYTECODE=1` evita `.pyc`; `TORCH_FORCE_WEIGHTS_ONLY_LOAD=1` reforca a politica se algum codigo tentar carregar algo. Nenhum caminho de checkpoint e fornecido. Entry points que executam trabalho/alteram estado durante import (por exemplo `train.train`, `train.preprocess` e os extratores CLI) sao compilados acima, mas deliberadamente nao importados.

In [ ]:
if not RVC_ROOT.is_dir():
    print("IMPORT TESTS: BLOCKED; checkout missing")
else:
    smoke_modules = [
        "train.data_utils",
        "train.losses",
        "train.mel_processing",
        "train.process_ckpt",
        "train.utils",
        "train.dataset.slicer2",
        "infer.module.commons",
        "infer.module.models",
        "infer.hubert",
        "infer.audio",
    ]
    smoke_code = "import importlib,sys; importlib.import_module(sys.argv[1]); print('IMPORT_OK', sys.argv[1])"
    smoke_env = os.environ.copy()
    smoke_env["CUDA_VISIBLE_DEVICES"] = "0"
    smoke_env["PYTHONDONTWRITEBYTECODE"] = "1"
    smoke_env["TORCH_FORCE_WEIGHTS_ONLY_LOAD"] = "1"
    smoke_env["HF_HUB_OFFLINE"] = "1"
    smoke_env["TRANSFORMERS_OFFLINE"] = "1"
    existing_pythonpath = smoke_env.get("PYTHONPATH", "")
    smoke_env["PYTHONPATH"] = str(RVC_ROOT) + (os.pathsep + existing_pythonpath if existing_pythonpath else "")
    import_results = []
    for module_name in smoke_modules:
        result = subprocess.run(
            [sys.executable, "-c", smoke_code, module_name],
            cwd=RVC_ROOT,
            env=smoke_env,
            capture_output=True,
            text=True,
            check=False,
        )
        passed = result.returncode == 0
        import_results.append((module_name, passed))
        print(("PASS" if passed else "FAIL") + ":", module_name)
        if result.stdout:
            print(result.stdout.strip())
        if result.stderr:
            print(result.stderr[-4000:])
        print("exit code:", result.returncode)
    print("IMPORT SUMMARY:", sum(passed for _, passed in import_results), "/", len(import_results), "passed")
    print("Import success alone is not full training or numerical compatibility.")

## Plano reproduzivel (sem iniciar treino)

1. Manter a imagem Kaggle testada (Python 3.13.15, PyTorch 2.11.0+cu128) sem trocar Torch/Torchaudio.
2. Definir `CUDA_VISIBLE_DEVICES=0` antes da primeira importacao/inicializacao CUDA; o notebook faz isso e usa apenas `cuda:0` visivel.
3. Clonar `RVC-Project/Retrieval-based-Voice-Conversion-WebUI`, fazer checkout do commit `81eed5e8f68b6bed1789f682fe78cdd324495afc` e verificar `git rev-parse HEAD` contra esse SHA.
4. Verificar que os patches revisados do HYRON para `torch.load(..., weights_only=True)` e workers configuraveis ja estao aplicados. A celula de auditoria nao aplica patches; nao remover nem contornar o carregamento restrito.
5. Nao instalar o requirements upstream Py3.12/cu128. Inventariar dependencias ja presentes, rodar `pip check`, compilacao estatica e imports isolados do caminho de treino; qualquer dependencia incompatível deve bloquear a tentativa, sem alterar Torch no ambiente base.
6. So apos demonstrar compatibilidade, anexar dataset proprio/autorizado e exigir confirmacao explicita antes de qualquer treino. Essa celula/notebook nao inicia treinamento.

A revisao local confirmou o commit e os patches restritos no checkout HYRON, mas nao executou o runtime em Python 3.13/PyTorch 2.11. O plano e reproduzivel; compatibilidade de treino permanece nao verificada.

## Proxima etapa manual

Quando a sessao estiver habilitada, o operador deve validar a autoria do dataset e confirmar explicitamente antes de executar qualquer passo de treinamento.

Checklist:
- GPU disponivel
- RAM e VRAM suficientes
- Dataset proprio ou explicitamente autorizado
- Configuracao ajustada sem usar `weights_only=False`
- Checkpoint validado antes de exportacao
